# Concurrency — downloading 20 days of prices, sequentially vs all at once.  ~5 min

Real network I/O: we pull **20 daily price files** from Binance's public archive. Downloading them one-by-one *waits* 20 times. Fill one blank to overlap the waits — and see why *"threads are useless in Python"* is wrong for I/O.

*(The sequential download is slow on purpose — that's the point. ~20 s.)*

In [ ]:
import urllib.request, zipfile, io, time
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor

SYMBOL = 'BTCUSDT'
days = [date(2024, 6, 1) + timedelta(d) for d in range(20)]   # 20 daily files

def close_of(day):
    """Download ONE daily kline file from Binance and return its close price."""
    url = f'https://data.binance.vision/data/spot/daily/klines/{SYMBOL}/1d/{SYMBOL}-1d-{day}.zip'
    raw = urllib.request.urlopen(url, timeout=20).read()
    z   = zipfile.ZipFile(io.BytesIO(raw))
    row = z.read(z.namelist()[0]).decode().split('\n')[0]
    return float(row.split(',')[4])          # column 4 = close price

In [ ]:
# sequential — download one file, wait, next… (slow on purpose)
t0 = time.perf_counter()
closes_seq = [close_of(d) for d in days]
t_seq = time.perf_counter() - t0
print(f'sequential : {t_seq:5.1f} s   ({len(days)} files, one after another)')

**Your turn.** Download the same 20 files, but **all at once**, on a `ThreadPoolExecutor`.

In [ ]:
def download_all(days):
    ## SOLUTION: map close_of over days on a ThreadPoolExecutor (max_workers=len(days)) ##
    with ThreadPoolExecutor(max_workers=len(days)) as ex:
        return list(ex.map(close_of, days))
    ## END ##

In [ ]:
t0 = time.perf_counter()
closes = download_all(days)
t_conc = time.perf_counter() - t0
assert closes == closes_seq, 'same files, same prices'
print(f'concurrent : {t_conc:5.1f} s')
print(f'\n→ ~{t_seq/t_conc:.0f}× faster — the downloads WAIT on the network, and the waits overlap.')
print('  The GIL is RELEASED during I/O, so plain threads are enough here.')

### And here are the returns

Now that we have the 20 daily closes, the daily returns and their cumulative product — the reason we downloaded anything at all.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

closes  = np.array(closes)
returns = closes[1:] / closes[:-1] - 1          # daily returns
cum     = (1 + returns).cumprod()               # 1 unit invested on day 1

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(range(1, len(closes)), 100*returns,
          color=['#4c9f70' if r >= 0 else '#d1495b' for r in returns])
ax[0].axhline(0, color='k', lw=.6); ax[0].set(title=f'{SYMBOL} daily return (%)', xlabel='day')
ax[1].plot(cum, marker='o', ms=3); ax[1].set(title='cumulative return', xlabel='day')
plt.tight_layout(); plt.show()